In [1]:
import requests

url = "https://www.fixmystreet.com/open311/v2/requests.json"
params = {
    "jurisdiction_id": "fixmystreet",
    "start_date": "2026-09-01T00:00:00Z",
    "end_date": "2026-09-02T00:00:00Z",
}

response = requests.get(url, params=params, timeout=30)
print(response.status_code)


200


In [2]:
data = response.json()
reports = data["service_requests"]

print(len(reports))
print(list(reports[0].keys()))
print([r["service_name"] for r in reports[:5]])


1000
['media_url', 'service_request_id', 'detail', 'interface_used', 'agency_responsible', 'agency_sent_datetime', 'requested_datetime', 'long', 'title', 'lat', 'service_code', 'comment_count', 'description', 'service_name', 'status', 'updated_datetime']
['Pothole on a road - More than 150mm wide and 40mm deep', 'Blocked Drain', 'Graffiti or Flyposting (not offensive)', 'Street Cleansing', 'Graffiti']


In [6]:
services_url = "https://www.fixmystreet.com/open311/v2/services.json"
response = requests.get(services_url, params={"jurisdiction_id": "fixmystreet"}, timeout=30)
print(response.status_code)

services = response.json()["services"]
print(len(services))
print(list(services[0].keys()))

200
2601
['description', 'type', 'metadata', 'service_name', 'service_code']


In [8]:
import re

animal_words = r"\b(dogs?|animals?|dead|fouling|fox|foxes|birds?|pigeons?|gulls?|rats?|rodents?|mice|vermin|pests?|horses?|wildlife|stray|carcass|carcasses|badgers?|deer|swans?|cats?)\b"
not_animal_words = r"\b(trees?|shrubs?|cats? eyes?|reflectors?)\b"

animal_services = [
    s for s in services
    if re.search(animal_words, s["service_name"], re.IGNORECASE)
    and not re.search(not_animal_words, s["service_name"], re.IGNORECASE)
]

print(len(animal_services))
for name in sorted({s["service_name"] for s in animal_services}):
    print(name)

73
Animal Licensing
Animal Nuisance
Animal signage
Bird fouling or insects
Carcass - Large
Carcass - Small
Clinical Waste - Dead Animal (Estates)
Clinical Waste - Dead Animal (Street)
Damaged dog bin
Damaged dog waste bin, rubbish bin or park bench
Dangerous Dog
Dangerous or intimidating animals
Dead Animal
Dead Animal (TS)
Dead Animal in Park
Dead Animal in a Park or Open Space
Dead Animals
Dead Animals on Road
Dead Dog
Dead animal
Dead animal (off the road)
Dead animal (on the road)
Dead animal in the carriageway or footway
Dead animal on a road or footpath
Dead animal that needs removing
Dead animals
Dog Bin Overflowing
Dog Bin overflow
Dog Fouling
Dog Fouling - Report a Location
Dog Fouling - Witnessed Incident
Dog Fouling in a Park or Open Space
Dog Mess
Dog Poo/Fouling
Dog Related Incident
Dog Waste - Signs
Dog Waste Bin - Empty
Dog Waste Bin - Emptying
Dog Waste Clearance (Estates)
Dog Waste Clearance (Street)
Dog and litter bins
Dog bin
Dog bin overflowing
Dog bin/fouling
Dog f

In [9]:
import pandas as pd

categories = pd.DataFrame(animal_services)[["service_code", "service_name"]]
categories.to_csv("../animal_categories.csv", index=False)
print(categories.shape)

(73, 2)


In [10]:
import time

reports_url = "https://www.fixmystreet.com/open311/v2/requests.json"

def fetch_reports(service_code, start_date, end_date):
    params = {
        "jurisdiction_id": "fixmystreet",
        "service_code": service_code,
        "start_date": start_date,
        "end_date": end_date,
    }
    response = requests.get(reports_url, params=params, timeout=30)
    response.raise_for_status()
    reports = response.json()["service_requests"]
    if len(reports) == 1000:
        print(f"Warning: limit of 1,000 reached for {service_code} from {start_date} – reports may be missing")
    return reports
    

In [11]:
test_codes = categories.loc[categories["service_name"] == "Dog fouling", "service_code"]

for code in test_codes:
    reports = fetch_reports(code, "2026-09-01T00:00:00Z", "2026-10-01T00:00:00Z")
    print(code, len(reports))
    time.sleep(1)

Dog fouling 724


In [12]:
september = []

for code in categories["service_code"]:
    reports = fetch_reports(code, "2026-09-01T00:00:00Z", "2026-10-01T00:00:00Z")
    september.extend(reports)
    time.sleep(1)

print(len(september))
print(pd.Series([r["service_name"] for r in september]).value_counts().head(10))

2184
Dog fouling                                724
Dead Animal                                252
Dead animal                                246
Dog Fouling                                214
Dead animals                               118
Dead animal (off the road)                  84
Dog Poo/Fouling                             74
Street cleaning - Dead animal on street     67
Overflowing litter/dog bin                  49
Street cleaning - Dog mess                  46
Name: count, dtype: int64


In [13]:
import json
from pathlib import Path

data_folder = Path("../data")
month_starts = pd.date_range("2025-10-01", "2026-10-01", freq="MS")

for start, end in zip(month_starts[:-1], month_starts[1:]):
    file = data_folder / f"reports_{start:%Y-%m}.json"
    if file.exists():
        print(file.name, "already there – skipped")
        continue
    month_reports = []
    for code in categories["service_code"]:
        month_reports.extend(fetch_reports(code, f"{start:%Y-%m-%d}T00:00:00Z", f"{end:%Y-%m-%d}T00:00:00Z"))
        time.sleep(1)
    file.write_text(json.dumps(month_reports))
    print(file.name, len(month_reports))

reports_2025-10.json 1439
reports_2025-11.json 1182
reports_2025-12.json 1164
reports_2026-01.json 1735
reports_2026-02.json 1786
reports_2026-03.json 1890
reports_2026-04.json 1963
reports_2026-05.json 1835
reports_2026-06.json 1707
reports_2026-07.json 1922
reports_2026-08.json 2122
reports_2026-09.json 2184
